## 3 Way convo

In [1]:
from openai import OpenAI
from IPython.display import Markdown, display

In [13]:
def husband_prompt(conversation_history):
    return f"""
    Your name is Mr. Ares. and you're the husband to Mrs. Ares. both you and your wife are trying to convince the car dealer to sell a car to your wife at a good price.

    this is the conversation history:
    {conversation_history}
    """

def wife_prompt(conversation_history):
    return f"""
    Your name is Mrs. Ares. and you're the wife to Mr. Ares. both you and your husband are trying to convince the car dealer to sell a car to you at a good price.

    this is the conversation history:
    {conversation_history}
    """

def car_dealer_prompt(conversation_history):
    return f"""
    Your name is Mr. Kenny the car dealer. and you're the car dealer. you're trying to sell a car to Mr. and Mrs. Ares.

    this is the conversation history:
    {conversation_history}
    """


system_prompt = """this is a conversation between a husband, wife and a car dealer. the husband and wife are trying to convince the car dealer to sell a car to the wife at a good price. the car dealer is trying to sell a car to the husband and wife at a good price.
It is a 3way conversations and one person should be direction their question to someone else

response format:

    {
        "role": "husband",
        "content": "question"
        "directed_to": "dealer or wife or husband"
    }

"""



In [19]:
openai = OpenAI(base_url="http://localhost:11434/v1")

In [8]:
!ollama pull llama3.2:1b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest 
pulling 74701a8c35f6: 100% ▕██████████████████▏ 1.3 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 4f659a1e86d7: 100% ▕██████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [24]:
import json
import re


def _parse_message_content(raw: str):
    """Extract a single JSON object from model output (handles extra text or markdown)."""
    text = (raw or "").strip()
    # Strip markdown code blocks if present
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        if "Extra data" in str(e) or e.pos:
            # Take only the first complete {...} object
            start = text.find("{")
            if start == -1:
                raise
            depth = 0
            for i in range(start, len(text)):
                if text[i] == "{":
                    depth += 1
                elif text[i] == "}":
                    depth -= 1
                    if depth == 0:
                        return json.loads(text[start : i + 1])
        raise


def get_response(prompt):
    response = openai.chat.completions.create(
        model="llama3.2:1b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    )
    content = response.choices[0].message.content
    message = _parse_message_content(content)
    return message





In [26]:
messages = []

while len(messages) < 10:
    if not messages:
        prompt = husband_prompt(messages)
    else:
        last = messages[-1]
        directed_to = last.get("directed_to", "")
        if directed_to == "dealer":
            prompt = car_dealer_prompt(messages)
        elif directed_to == "wife":
            prompt = wife_prompt(messages)
        else:
            prompt = husband_prompt(messages)
    message = get_response(prompt)
    messages.append(message)
    print(message)


{'role': 'wife', 'content': 'I would like to take out a loan and drive that blue sedan, but my husband insists on taking a 5-year-old model.', 'directed_to': None}


JSONDecodeError: Expecting value: line 1 column 1 (char 0)